In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import torch

In [2]:
def get_pca_anomaly_score(X, n_components=3):
    """
    PCA Reconstruction Error.
    Fits PCA on the current cross-section of stocks. 
    High MSE = market dynamics for this stock are detached from main linear factors.
    """
    # Always scale before PCA
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    pca = PCA(n_components=n_components)
    pca.fit(X_scaled)
    
    # Transform and reconstruct
    X_reconstructed = pca.inverse_transform(pca.transform(X_scaled))
    
    # Calculate Mean Squared Error per stock
    mse = np.mean((X_scaled - X_reconstructed)**2, axis=1)
    
    # Normalize to [0, 1] for easy comparison with your GAT signals
    return (mse - mse.min()) / (mse.max() - mse.min() + 1e-8) # size N_stocks 

In [3]:
print("\n[1/6] Loading data...")
device = torch.device("cpu")

# Load prices
prices = pd.read_excel('SPX_sectors_data.xlsx', header=[0,1], index_col=0)
prices.dropna(how='all', inplace=True)
prices = prices.ffill().bfill()
prices.columns = prices.columns.droplevel(1)

all_stocks = prices.columns.tolist()
stock2idx = {s: i for i, s in enumerate(all_stocks)}
idx2stock = {i: s for s, i in stock2idx.items()}
N_full = len(all_stocks)

# Load sectors
sectors = pd.read_excel('SPX_sectors_data.xlsx', sheet_name='Sectors', 
                        header=0, index_col=0)
sectors['sector_id'] = sectors['Sector'].astype('category').cat.codes


tickers = prices.columns.get_level_values(0).unique().tolist()
node_map = {ticker: i for i, ticker in enumerate(tickers)}

# Split data: Train 2007-2009, Test 2019-2021
train_prices = prices.loc['2012-01-01':'2016-12-31']
test_prices = prices.loc['2019-07-01':'2024-12-31']
val_prices = prices.loc["2017-01-01":"2019-06-30"]

returns = pd.read_excel('SPX_sectors_data.xlsx', header=[0,1], index_col=0)
returns.columns = returns.columns.get_level_values(0)
returns.dropna(how='all', inplace=True) 
returns = returns.pct_change().dropna(how='all')
#returns = returns.ffill().bfill()
# Compute returns
# train_returns = train_prices.pct_change().dropna(how='all')
# test_returns = test_prices.pct_change().dropna(how='all')
train_returns = returns.loc['2012-01-01':'2016-12-31']
train_returns = train_returns.ffill().bfill()
test_returns = returns.loc['2019-07-01':'2024-12-31']
test_returns = test_returns.ffill().bfill()
val_returns = returns.loc["2017-01-01":"2019-06-30"]
val_dates = val_returns.index
all_dates = sorted(set(train_returns.index) | set(test_returns.index) | set(val_returns.index))
test_dates = test_returns.index
# Compute volatility
train_volatility = train_returns.rolling(window=21).std().dropna(how='all') * np.sqrt(252)
test_volatility = test_returns.rolling(window=21).std().dropna(how='all') * np.sqrt(252)
volatility = pd.concat([train_volatility, test_volatility]).sort_index()

# Helper to load and fix dates
def load_and_fix_index(filename):
    df = pd.read_csv(filename, index_col=0)
    df.index = pd.to_datetime(df.index, format='%m/%d/%Y') # Fix date format
    df.dropna(how='all', inplace=True)
    df = df.ffill().bfill()
    return df

# Load Constituent Factors (Tables where Cols = Tickers, Rows = Dates)
Market_caps = load_and_fix_index('Data/SPX_Constituents_market_cap_2006_2025(in).csv')

PE_ratios = load_and_fix_index('Data/SPX_Constituents_Calculated_PE_2006_2025(in).csv')

Implied_vol = load_and_fix_index('Data/SPX_Constituents_Implied_vol_2006_2025(in).csv')

Beta = load_and_fix_index('Data/SPX_Constituents_Beta_2006_2025(in).csv')
Operating_margin = load_and_fix_index('Data/SPX_Constituents_Op_Margin_2006_2025(in).csv')   
Return_on_equity = load_and_fix_index('Data/SPX_Constituents_Ret_On_Equity_2006_2025(in).csv')
RSI_momentum = load_and_fix_index('Data/SPX_Constituents_RSI_momentum_2006_2025(in).csv')
Short_interest = load_and_fix_index('Data/SPX_Constituents_Short_Interest_Pct_2006_2025(in).csv')
Turnover = load_and_fix_index('Data/SPX_Constituents_Turnover_30D_2006_2025(in).csv')


[1/6] Loading data...


C:\Users\archi\AppData\Local\Temp\ipykernel_36084\1677540423.py:32: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = returns.pct_change().dropna(how='all')


In [4]:
print('Precomputing Z scores for all features...')

def precompute_zscores(df, window=63):
    """Calculates rolling z-scores for an entire dataframe at once."""
    rolling_mean = df.rolling(window=window, min_periods=5).mean()
    rolling_std = df.rolling(window=window, min_periods=5).std()
    # Avoid division by zero with 1e-8
    z_scores = (df - rolling_mean) / (rolling_std + 1e-8)
    # Clip outliers to keep gradients stable and fill NaNs
    return z_scores.clip(-5.0, 5.0).fillna(0.0)

# Precompute z-scores for all feature dataframes
Z_DATA = {
    'volatility':       precompute_zscores(volatility), # use your vol_window here
    'market_caps':      precompute_zscores(Market_caps),
    'pe_ratios':        precompute_zscores(PE_ratios),
    'implied_vol':      precompute_zscores(Implied_vol),
    'short_interest':   precompute_zscores(Short_interest),
    'beta':             precompute_zscores(Beta),
    'op_margin':        precompute_zscores(Operating_margin),
    'roe':              precompute_zscores(Return_on_equity),
    'rsi':              precompute_zscores(RSI_momentum),
    'turnover':         precompute_zscores(Turnover)
}

Precomputing Z scores for all features...


In [5]:

def prepare_node_features(stocks, sectors, Z_DATA, t, norm_window=63): # 1 quarter
    """
    Normalise each feature per stock against its own rolling history (z-score),
    preserving signal relative to that stock's recent behaviour.
    """
    rows = []
    t = pd.to_datetime(t)
    
    for stock in stocks:
        sector_id = sectors.loc[stock, 'sector_id'] if stock in sectors.index else 0
        
        # Simple lookup instead of rolling calculation
        feats = [sector_id]
        for key in Z_DATA:
            df = Z_DATA[key]
            val = df.loc[t, stock] if stock in df.columns and t in df.index else 0.0
            feats.append(float(val))
            
        rows.append(feats)

    features = np.array(rows, dtype=np.float32)
    return torch.tensor(np.nan_to_num(features), dtype=torch.float32)

In [6]:

def get_active_stocks(returns, t, lookback_days, feature_dfs=None, min_obs=21, eps=0.0):
    t = pd.to_datetime(t)
    window = returns.loc[t - pd.Timedelta(days=lookback_days): t]
    #print(f"Window for active stock selection: {window.index.min().date()} to {window.index.max().date()}")
    # enough non-NaN observations
    counts = window.notna().sum(axis=0)
    ok_obs = counts >= min_obs

    # not constant zero in the window (treat as missing asset)
    if eps == 0.0:
        ok_nonzero = ~(window.fillna(0.0) == 0.0).all(axis=0)
    else:
        ok_nonzero = ~(window.fillna(0.0).abs() <= eps).all(axis=0)

    #active = window.columns[ok_obs & ok_nonzero].tolist()
    active = window.columns[ok_nonzero].tolist()
    #print(f"Active stocks before feature intersection: {len(active)}")
    if feature_dfs is not None:
        active_set = set(active)
        for df in feature_dfs:
            # only care if the stock exists as a column in the dataframe
            if not df.empty:
                # Find intersection between current active stocks and this dataframe's columns
                active_set = active_set.intersection(df.columns)
        
        active = list(active_set)

    return active

In [7]:
# want error per stock per time step 
results = {} # date x stock with PCA anomaly score
K = 21
for t in test_dates:
    stocks = get_active_stocks(returns, t, lookback_days=K, feature_dfs=[volatility, Market_caps, PE_ratios, Implied_vol, Short_interest, Beta, Operating_margin, Return_on_equity, RSI_momentum, Turnover], min_obs=21)
    X = prepare_node_features(stocks, sectors, Z_DATA, t, norm_window=63) # shape (N_stocks, N_features)
    #print(X.shape)

    signals = get_pca_anomaly_score(X.numpy(), n_components=3)
    results[t] = stocks, signals.reshape(-1) # signals is now shape (N,)


torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size([459, 11])
torch.Size

In [11]:
def evaluate_once(
        test_results,
        prices,
        forward_window,
        crash_threshold
):

    y_true = []
    y_scores = []

    sorted_dates = sorted(test_results.keys())

    valid_dates = [
        d for d in sorted_dates
        if d <= prices.index[-1] - pd.Timedelta(days=forward_window)
    ]

    for t in valid_dates:

        stocks, signals = test_results[t]

        if isinstance(signals, torch.Tensor):
            signals = signals.cpu().numpy()

        signals = signals.flatten()

        available = [s for s in stocks if s in prices.columns]

        if len(available) == 0:
            continue

        mask = [i for i,s in enumerate(stocks) if s in available]

        signals = signals[mask]

        p_t = prices.loc[t, available]

        future_idx = prices.index.searchsorted(
            t + pd.Timedelta(days=forward_window)
        )

        if future_idx >= len(prices):
            continue

        future_date = prices.index[future_idx]

        p_future = prices.loc[future_date, available]

        fwd_returns = (p_future - p_t) / p_t

        crash = (fwd_returns < crash_threshold).astype(int)

        y_true.extend(crash.values)

        y_scores.extend(signals)

    if len(y_true) == 0:
        return None

    y_true = np.array(y_true)
    y_scores = np.array(y_scores)

    auc = roc_auc_score(y_true, y_scores)

    baseline = y_true.mean()

    precision = y_true[y_scores > np.percentile(y_scores, 90)].mean()

    lift = precision / baseline if baseline > 0 else np.nan

    return auc, lift, baseline

def grid_search(
        test_results,
        prices,
        forward_windows,
        crash_thresholds
):

    rows = []

    for fw in forward_windows:

        for ct in crash_thresholds:

            result = evaluate_once(
                test_results,
                prices,
                fw,
                ct
            )

            if result is None:
                continue

            auc, lift, baseline = result

            rows.append({

                "ForwardWindow": fw,

                "CrashThreshold": ct,

                "AUC": auc,

                "Lift": lift,

                "Baseline": baseline

            })

            print(
                f"FW={fw:3d} "
                f"CT={ct:6.2f} "
                f"AUC={auc:.3f} "
                f"Lift={lift:.2f}"
            )

    return pd.DataFrame(rows)

In [12]:
forward_windows = [10,22,44]

crash_thresholds = [-0.10,-0.15,-0.20,-0.30]

df_results = grid_search(

    results,

    test_prices,

    forward_windows,

    crash_thresholds

)
# for test_results_test : FW=  5 CT= -0.30 AUC=0.785 Lift=4.45 testing from 2020-2024


FW= 10 CT= -0.10 AUC=0.510 Lift=1.10
FW= 10 CT= -0.15 AUC=0.495 Lift=1.06
FW= 10 CT= -0.20 AUC=0.474 Lift=0.95
FW= 10 CT= -0.30 AUC=0.457 Lift=1.04
FW= 22 CT= -0.10 AUC=0.524 Lift=1.12
FW= 22 CT= -0.15 AUC=0.528 Lift=1.13
FW= 22 CT= -0.20 AUC=0.512 Lift=1.07
FW= 22 CT= -0.30 AUC=0.465 Lift=0.92
FW= 44 CT= -0.10 AUC=0.545 Lift=1.22
FW= 44 CT= -0.15 AUC=0.559 Lift=1.32
FW= 44 CT= -0.20 AUC=0.580 Lift=1.47
FW= 44 CT= -0.30 AUC=0.604 Lift=1.60
